<a href="https://colab.research.google.com/github/wikriwiki/dacon_1/blob/main/dacon_best.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DACON 구조물 안정성 추론 — 최고 성능 버전
**개선 사항 요약:**
- ✅ Train Augmentation (Flip, ColorJitter, RandomRotation, RandomErasing)
- ✅ Label Smoothing BCE Loss
- ✅ Early Stopping (patience=5)
- ✅ Warmup + CosineAnnealing 스케줄러
- ✅ Gradient Clipping
- ✅ Train+Dev 합산 재학습 (최종 제출 전)
- ✅ TTA (수평/수직 반전 앙상블)
- ✅ Classifier Head 단순화 (과적합 억제)

In [ ]:
!pip install -q transformers accelerate timm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import zipfile

DRIVE_ROOT = "/content/drive/MyDrive/dacon"
ZIP_PATH = os.path.join(DRIVE_ROOT, "data.zip")
EXTRACT_ROOT = os.path.join(DRIVE_ROOT, "data_unzipped")

os.makedirs(EXTRACT_ROOT, exist_ok=True)

if not os.path.exists(os.path.join(EXTRACT_ROOT, "train.csv")):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_ROOT)
    print("압축 해제 완료")
else:
    print("이미 압축 해제됨, 스킵")

In [ ]:
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import AutoImageProcessor, AutoModel
from sklearn.metrics import accuracy_score, roc_auc_score
from tqdm.auto import tqdm

# ── 하이퍼파라미터 ──────────────────────────────────────
SEED         = 42
IMG_SIZE     = 224
BATCH_SIZE   = 8
NUM_EPOCHS   = 30          # Early stopping이 알아서 멈춰줌
NUM_WORKERS  = 2
PATIENCE     = 5           # dev AUC 개선 없으면 5 에폭 후 종료
WARMUP_EPOCHS = 2          # 처음 2 에폭은 LR warmup
LABEL_SMOOTH  = 0.05       # Label smoothing 강도
GRAD_CLIP     = 1.0        # Gradient clipping
DROPOUT       = 0.3        # 과적합 억제

MODEL_NAME   = "facebook/dinov2-base"
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

DATA_ROOT    = EXTRACT_ROOT
TRAIN_CSV    = os.path.join(DATA_ROOT, "train.csv")
DEV_CSV      = os.path.join(DATA_ROOT, "dev.csv")
SAMPLE_SUB_CSV = os.path.join(DATA_ROOT, "sample_submission.csv")
TRAIN_DIR    = os.path.join(DATA_ROOT, "train")
DEV_DIR      = os.path.join(DATA_ROOT, "dev")
TEST_DIR     = os.path.join(DATA_ROOT, "test")
SAVE_DIR     = os.path.join(DRIVE_ROOT, "checkpoints_best")
os.makedirs(SAVE_DIR, exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
print("DEVICE:", DEVICE)

In [ ]:
# ── Processor 로드 & 정규화값 추출 ──────────────────────
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
image_mean = processor.image_mean
image_std  = processor.image_std

# Train: Augmentation 포함
train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.3),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    T.RandomRotation(degrees=15),
    T.ToTensor(),
    T.Normalize(mean=image_mean, std=image_std),
    T.RandomErasing(p=0.2, scale=(0.02, 0.1)),  # 랜덤 패치 지우기
])

# Val / Test: 변환 없음
val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=image_mean, std=image_std),
])

print("Transforms 준비 완료")

In [ ]:
# ── Dataset ─────────────────────────────────────────────
class MultiViewDataset(Dataset):
    def __init__(self, df, image_root, transform=None, is_test=False):
        self.df         = df.reset_index(drop=True)
        self.image_root = image_root
        self.transform  = transform
        self.is_test    = is_test

    def __len__(self):
        return len(self.df)

    def _load_image(self, sample_id, view_name):
        path = os.path.join(self.image_root, str(sample_id), f"{view_name}.png")
        return Image.open(path).convert("RGB")

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        sample_id = row["id"]
        front_img = self._load_image(sample_id, "front")
        top_img   = self._load_image(sample_id, "top")

        if self.transform:
            front_img = self.transform(front_img)
            top_img   = self.transform(top_img)

        item = {"id": sample_id, "front": front_img, "top": top_img}
        if not self.is_test:
            item["target"] = torch.tensor(row["target"], dtype=torch.float32)
        return item

In [ ]:
# ── 데이터 로드 ──────────────────────────────────────────
label_map = {"stable": 0, "unstable": 1}

train_df = pd.read_csv(TRAIN_CSV)
dev_df   = pd.read_csv(DEV_CSV)
sample_sub = pd.read_csv(SAMPLE_SUB_CSV)

train_df["target"] = train_df["label"].map(label_map)
dev_df["target"]   = dev_df["label"].map(label_map)

# dev 이미지 경로를 TRAIN_DIR 기준으로 맞추기 위한 image_root 분리
train_dataset = MultiViewDataset(train_df, TRAIN_DIR, train_transform)
dev_dataset   = MultiViewDataset(dev_df,   DEV_DIR,   val_transform)
test_dataset  = MultiViewDataset(sample_sub, TEST_DIR, val_transform, is_test=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
dev_loader   = DataLoader(dev_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"train: {len(train_df)}, dev: {len(dev_df)}, test: {len(sample_sub)}")

In [ ]:
# ── Label Smoothing Loss ─────────────────────────────────
class LabelSmoothingBCELoss(nn.Module):
    """BCEWithLogitsLoss + label smoothing으로 과신 방지"""
    def __init__(self, smoothing=0.05):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, logits, targets):
        # 0→smoothing/2, 1→1-smoothing/2 로 부드럽게
        smooth_targets = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        return F.binary_cross_entropy_with_logits(logits, smooth_targets)

In [ ]:
# ── 모델 ─────────────────────────────────────────────────
class DINOv2MultiViewClassifier(nn.Module):
    def __init__(self, model_name="facebook/dinov2-base", dropout=0.3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size   = self.backbone.config.hidden_size  # 768

        # 단순화된 head: 과적합 억제 + BatchNorm으로 안정화
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 4, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 1)
        )

    def encode_image(self, x):
        return self.backbone(pixel_values=x).last_hidden_state[:, 0]  # CLS token

    def forward(self, front, top):
        f_front = self.encode_image(front)
        f_top   = self.encode_image(top)

        # 4가지 관계 표현 concat (원본 전략 유지)
        fused = torch.cat([
            f_front,
            f_top,
            torch.abs(f_front - f_top),
            f_front * f_top,
        ], dim=1)

        return self.classifier(fused).squeeze(1)

In [ ]:
# ── 옵티마이저 & 스케줄러 ────────────────────────────────
model = DINOv2MultiViewClassifier(MODEL_NAME, dropout=DROPOUT).to(DEVICE)

backbone_params = [p for n, p in model.named_parameters() if "backbone" in n]
head_params     = [p for n, p in model.named_parameters() if "backbone" not in n]

optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": 1e-5},
    {"params": head_params,     "lr": 1e-4},
], weight_decay=1e-4)

# Warmup: 처음 WARMUP_EPOCHS는 LR을 0에서 선형으로 올림
def warmup_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    return 1.0

warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_lambda)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS - WARMUP_EPOCHS, eta_min=1e-7
)

criterion = LabelSmoothingBCELoss(smoothing=LABEL_SMOOTH)

print("모델 준비 완료")

In [ ]:
# ── 학습 함수 ────────────────────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, total=len(loader))
    for batch in pbar:
        front  = batch["front"].to(device)
        top    = batch["top"].to(device)
        target = batch["target"].to(device)

        optimizer.zero_grad()
        logits = model(front, top)
        loss   = criterion(logits, target)
        loss.backward()

        # Gradient clipping: 폭발적 업데이트 방지
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        total_loss += loss.item() * front.size(0)
        pbar.set_description(f"train loss: {loss.item():.4f}")

    return total_loss / len(loader.dataset)


def LOGLOSS(true, pred, eps=1e-15):
    pred = np.clip(pred, eps, 1 - eps)
    pred = pred / np.sum(pred, axis=1).reshape(-1, 1)
    loss = -np.sum(true * np.log(pred), axis=1)
    return np.mean(loss)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_targets, all_probs = 0.0, [], []
    with torch.no_grad():
        for batch in loader:
            front  = batch["front"].to(device)
            top    = batch["top"].to(device)
            target = batch["target"].to(device)
            logits = model(front, top)
            loss   = criterion(logits, target)
            probs  = torch.sigmoid(logits)
            total_loss += loss.item() * front.size(0)
            all_targets.extend(target.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)
    preds    = [1 if p >= 0.5 else 0 for p in all_probs]
    acc      = accuracy_score(all_targets, preds)
    try:
        auc = roc_auc_score(all_targets, all_probs)
    except ValueError:
        auc = None

    probs_2d   = np.stack([1 - np.array(all_probs), np.array(all_probs)], axis=1)
    targets_2d = np.stack([1 - np.array(all_targets), np.array(all_targets)], axis=1)
    logloss    = LOGLOSS(targets_2d, probs_2d)

    return avg_loss, acc, auc, logloss

In [ ]:
best_dev_logloss = 1e9
patience_cnt  = 0
best_path     = os.path.join(SAVE_DIR, "best_model.pt")
history       = []

for epoch in range(NUM_EPOCHS):
    print(f"\n===== Epoch {epoch+1}/{NUM_EPOCHS} =====")

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    dev_loss, dev_acc, dev_auc, dev_logloss = evaluate(model, dev_loader, criterion, DEVICE)

    if epoch < WARMUP_EPOCHS:
        warmup_scheduler.step()
    else:
        cosine_scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']
    auc_str = f"{dev_auc:.4f}" if dev_auc is not None else "N/A"
    print(f"train_loss: {train_loss:.4f} | dev_loss: {dev_loss:.4f} | "
          f"dev_acc: {dev_acc:.4f} | dev_auc: {auc_str} | dev_logloss: {dev_logloss:.4f} | lr: {current_lr:.2e}")

    history.append({
        "epoch": epoch + 1, "train_loss": train_loss,
        "dev_loss": dev_loss, "dev_acc": dev_acc, "dev_auc": dev_auc, "dev_logloss": dev_logloss
    })

    if dev_logloss < best_dev_logloss:
        best_dev_logloss = dev_logloss
        patience_cnt = 0
        torch.save(model.state_dict(), best_path)
        print(f"  ▶ Best model 저장 (LogLoss={best_dev_logloss:.4f})")
    else:
        patience_cnt += 1
        print(f"  patience: {patience_cnt}/{PATIENCE}")
        if patience_cnt >= PATIENCE:
            print("  Early stopping!")
            break

print(f"\n학습 완료. Best dev LogLoss: {best_dev_logloss:.4f}")

In [ ]:
# ── Train + Dev 합산 재학습 (최종 제출 직전) ─────────────
print("Train + Dev 합산으로 재학습 시작...")

# dev_df의 image_root 통일: dev 이미지를 train 폴더 구조와 맞게 사용
# 두 데이터셋을 별도 Dataset으로 만들어 ConcatDataset으로 합침
from torch.utils.data import ConcatDataset

train_dataset_all = MultiViewDataset(train_df, TRAIN_DIR, train_transform)
dev_dataset_aug   = MultiViewDataset(dev_df,   DEV_DIR,   train_transform)  # dev도 aug 적용
all_dataset       = ConcatDataset([train_dataset_all, dev_dataset_aug])
all_loader        = DataLoader(all_dataset, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True)

# Best model 로드 후 재학습
model.load_state_dict(torch.load(best_path, map_location=DEVICE))

# 재학습용 옵티마이저 (더 낮은 LR)
optimizer_final = torch.optim.AdamW([
    {"params": backbone_params, "lr": 5e-6},
    {"params": head_params,     "lr": 5e-5},
], weight_decay=1e-4)

# best_epoch 기준으로 동일 에폭 수만큼 재학습
best_epoch = len(history) - patience_cnt  # early stop 기준 best 에폭
retrain_epochs = max(best_epoch, 5)

for epoch in range(retrain_epochs):
    loss = train_one_epoch(model, all_loader, criterion, optimizer_final, DEVICE)
    print(f"  Retrain epoch {epoch+1}/{retrain_epochs} | loss: {loss:.4f}")

final_path = os.path.join(SAVE_DIR, "final_model.pt")
torch.save(model.state_dict(), final_path)
print("Final model 저장:", final_path)

In [ ]:
# ── TTA 추론 (수평/수직 반전 앙상블) ────────────────────
def tta_transforms(image_mean, image_std, img_size=224):
    """중력 방향 보존 TTA — 수직반전 제거"""
    normalize = [T.ToTensor(), T.Normalize(mean=image_mean, std=image_std)]
    return [
        # 원본
        T.Compose([T.Resize((img_size, img_size))] + normalize),
        # 수평반전 (좌우 대칭 — 중력 방향 유지되므로 안전)
        T.Compose([T.Resize((img_size, img_size)), T.RandomHorizontalFlip(p=1.0)] + normalize),
        # 밝기/대비 살짝 변형 (중립적)
        T.Compose([T.Resize((img_size, img_size)),
                   T.ColorJitter(brightness=0.1, contrast=0.1)] + normalize),
    ]


def predict_tta(model, df, image_root, image_mean, image_std, device, is_test=True):
    """TTA 앙상블 예측"""
    model.eval()
    transforms_list = tta_transforms(image_mean, image_std)
    all_probs = None

    for tfm in transforms_list:
        dataset = MultiViewDataset(df, image_root, tfm, is_test=is_test)
        loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)
        probs_run = []
        with torch.no_grad():
            for batch in tqdm(loader, desc="TTA inference"):
                front  = batch["front"].to(device)
                top    = batch["top"].to(device)
                logits = model(front, top)
                probs  = torch.sigmoid(logits).cpu().numpy().tolist()
                probs_run.extend(probs)

        probs_arr = np.array(probs_run)
        all_probs = probs_arr if all_probs is None else all_probs + probs_arr

    return all_probs / len(transforms_list)  # 평균


print("TTA 추론 시작...")
unstable_probs = predict_tta(
    model, sample_sub, TEST_DIR, image_mean, image_std, DEVICE
)

submission = sample_sub.copy()
submission["unstable_prob"] = unstable_probs
submission["stable_prob"]   = 1.0 - unstable_probs
print(submission.head())

In [ ]:
# ── 제출 파일 저장 ────────────────────────────────────────
sub_path = os.path.join(DRIVE_ROOT, "submission_best.csv")
submission.to_csv(sub_path, index=False, encoding="utf-8-sig")
print("저장 완료:", sub_path)
print(submission.describe())